In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import torchvision.models as models

import albumentations as A
from albumentations.pytorch import ToTensorV2

import time
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm import tqdm
import copy
import pandas as pd




Parameters

In [ ]:
LEARNING_RATE = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
NUM_EPOCHS = 150

# Set to True for continuous confidence scores (0.00 to 1.00)
# Set to False for binary output (0 or 1)
OUTPUT_CONFIDENCE_SCORES = False

# --- EXPERIMENT MODES (Turn only ONE on at a time) ---
RUN_ABLATION_STUDY = False       # 20 runs excluding 1 feature at a time
RUN_SINGLE_FEATURE_STUDY = False # 20 runs training with only 1 feature at a time
RUN_CUSTOM_FEATURE_STUDY = False  # Set to True to use your own custom subset of features
CUSTOM_FEATURES = ["coh_vh_diff", "coh_vv_diff", "dem", "coh_vh_nonflood", "coh_vh_flood", "coh_vv_nonflood", "coh_vv_flood", "amp_vh_reference", "amp_vh_secondary", "amp_vv_reference", "amp_vv_secondary", "slope", "streamflow_direction"] # Modify this list as desired

NUM_WORKERS = 4
IMAGE_HEIGHT = 318
IMAGE_WIDTH = 318
PIN_MEMORY = True
LOAD_MODEL = False
TRAIN_DIR = "/home/st-juho/code_testing/split_dataset/train"
VAL_DIR = "/home/st-juho/code_testing/split_dataset/val"

In [27]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # if you use multi-GPU
    
#     # Enforce deterministic behavior in cuDNN algorithms
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

Train and Val Shenanigans if Needed

In [28]:
# import os, shutil
# import numpy as np
# from PIL import Image
# from tqdm import tqdm

# SRC = "/home/st-juho/code_testing/split_dataset"
# DST = "/home/st-juho/code_testing/split_dataset_filtered"
# LRS = ["mask", "coh_vh_diff", "coh_vv_diff", "coh_vh_nonflood", "coh_vh_flood", "coh_vv_nonflood", "coh_vv_flood", 
#        "amp_vh_reference", "amp_vh_secondary", "amp_vv_reference", "amp_vv_secondary", "dem", "land_use", 
#        "buildings", "inlets", "outlets", "open_drains", "gravity_mains"]

# for splt in ["train", "val"]:
#     s_path, d_path = os.path.join(SRC, splt), os.path.join(DST, splt)
#     os.makedirs(d_path, exist_ok=True)
#     masks = [f for f in os.listdir(s_path) if f.endswith("_mask.png")]
    
#     print(f"Filtering {splt}...")
#     count = 0
#     for m in tqdm(masks):
#         if np.max(np.array(Image.open(os.path.join(s_path, m)))) > 0:
#             base = m.replace("_mask.png", "")
#             for l in LRS:
#                 f = f"{base}_{l}.png"
#                 if os.path.exists(os.path.join(s_path, f)):
#                     shutil.copy2(os.path.join(s_path, f), os.path.join(d_path, f))
#             count += 1
            
#     # Emergency move if val is empty
#     if splt == "val" and count == 0:
#         t_path = os.path.join(DST, "train")
#         tm = [f for f in os.listdir(t_path) if f.endswith("_mask.png")][0]
#         base = tm.replace("_mask.png", "")
#         for l in LRS:
#             shutil.move(os.path.join(t_path, f"{base}_{l}.png"), os.path.join(d_path, f"{base}_{l}.png"))
#         print(f"Moved {base} to val (val was empty)")

# print(f"Done. Filtered data at: {DST}")

Code

U-Net definitions

In [29]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)
    
class UNET(nn.Module):
    def __init__(self, in_channels=20, out_channels=1, features=[64, 128, 256, 512], 
                 land_use_idx=None, num_land_use_classes=256, embedding_dim=8):
        super(UNET, self).__init__()
        self.ups = nn.ModuleList()
        self.downs = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.land_use_idx = land_use_idx
        self.embedding_dim = embedding_dim
        
        # Calculate actual channel size entering the first Conv layer
        if self.land_use_idx is not None:
            self.land_use_embedding = nn.Embedding(num_land_use_classes, embedding_dim)
            # All continuous channels (in_channels - 1) + the new embedded channel vector depth
            actual_in_channels = (in_channels - 1) + embedding_dim
        else:
            actual_in_channels = in_channels

        # Down part of UNET
        for feature in features:
            self.downs.append(DoubleConv(actual_in_channels, feature))
            actual_in_channels = feature

        # Up part of UNET
        for feature in reversed(features):
            self.ups.append(
                nn.ConvTranspose2d(
                    feature*2, feature, kernel_size=2, stride=2,
                )
            )
            self.ups.append(DoubleConv(feature*2, feature))

        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        # If land_use layer exists in this run, parse and embed it
        if self.land_use_idx is not None:
            # 1. Isolate the land use channel directly as a 3D tensor -> Shape: [B, H, W]
            land_use_input = x[:, self.land_use_idx, :, :].long()
            
            # 2. Grab all other continuous layers -> Shape: [B, C-1, H, W]
            continuous_indices = [i for i in range(x.shape[1]) if i != self.land_use_idx]
            
            # 3. Process via lookup table -> Shape: [B, H, W, embedding_dim]
            embedded = self.land_use_embedding(land_use_input)
            # Permute back to standard PyTorch image layout -> Shape: [B, embedding_dim, H, W]
            embedded = embedded.permute(0, 3, 1, 2)
            
            if len(continuous_indices) > 0:
                continuous_channels = x[:, continuous_indices, :, :]
                # 4. Recombine features back together on the channel dimension
                x = torch.cat([continuous_channels, embedded], dim=1)
            else:
                # If land_use is the ONLY layer (Single Feature Study), x is just the embedding channels
                x = embedded
            
        skip_connections = []

        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip_connection = skip_connections[idx//2]

            if x.shape != skip_connection.shape:
                x = TF.resize(x, size=skip_connection.shape[2:])

            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.ups[idx+1](concat_skip)

        return self.final_conv(x)
    

def test():
    x = torch.randn((3,1,160,160))
    model = UNET(in_channels=1, out_channels=1)
    preds = model(x)
    print(preds.shape)
    print(x.shape)
    assert preds.shape == x.shape

if __name__ == "__main__":
    test()

torch.Size([3, 1, 160, 160])
torch.Size([3, 1, 160, 160])


Importing functions

In [30]:
class InSARDataset(Dataset):
    def __init__(self, root_dir, transform=None, exclude_idx=None, include_only_idx=None, custom_features=None):
        self.root_dir = root_dir
        self.transform = transform
        self.exclude_idx = exclude_idx
        self.include_only_idx = include_only_idx
        self.custom_features = custom_features # <-- ADD THIS LINE
        self.mask_files = sorted([f for f in os.listdir(root_dir) if f.endswith("_mask.png")])
        
        self.suffixes = [
            "coh_vh_diff", "coh_vv_diff", 
            "coh_vh_nonflood", "coh_vh_flood", "coh_vv_nonflood", "coh_vv_flood",
            "amp_vh_reference", "amp_vh_secondary", "amp_vv_reference", "amp_vv_secondary",
            "dem", "land_use", "buildings",
            "inlets", "outlets", "open_drains", "gravity_mains", "roads", "slope", "streamflow_direction"
        ]

    def __len__(self):
        return len(self.mask_files)

    def __getitem__(self, index):
        mask_filename = self.mask_files[index]
        base_tile = mask_filename.rsplit('_mask.png', 1)[0]
        
        layers = []
        
        # --- CASE A: CUSTOM FEATURE SELECTION ---
        if self.custom_features is not None:
            for s in self.custom_features:
                file_path = os.path.join(self.root_dir, f"{base_tile}_{s}.png")
                img = np.array(Image.open(file_path).convert("L"), dtype=np.float32)
                layers.append(img)
        # --- CASE B: STANDARD / ABLATION / SINGLE FEATURE MODES ---
        else:
            for i, s in enumerate(self.suffixes):
                if self.exclude_idx is not None and i == self.exclude_idx:
                    continue
                if self.include_only_idx is not None and i != self.include_only_idx:
                    continue
                    
                file_path = os.path.join(self.root_dir, f"{base_tile}_{s}.png")
                img = np.array(Image.open(file_path).convert("L"), dtype=np.float32)
                layers.append(img)

        image = np.stack(layers, axis=-1)
        
        mask_path = os.path.join(self.root_dir, mask_filename)
        mask = np.array(Image.open(mask_path).convert("L"), dtype=np.float32)
        mask[mask > 0] = 1.0 

        if self.transform is not None:
            augmentations = self.transform(image=image, mask=mask)
            image = augmentations['image']
            mask = augmentations['mask']

        return image, mask

Utility functions

In [31]:
def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state, filename)

def load_checkpoint(checkpoint, model):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint['state_dict'])

def get_loaders(
    train_dir,
    train_maskdir,
    val_dir,
    val_maskdir,
    batch_size,
    train_transform,
    val_transform,
    num_workers=4,
    pin_memory=True,
    exclude_idx=None,
    include_only_idx=None,
    custom_features=None # <-- ADD THIS LINE
):
    train_ds = InSARDataset(
        root_dir=train_dir,
        transform=train_transform,
        exclude_idx=exclude_idx,
        include_only_idx=include_only_idx,
        custom_features=custom_features # <-- PASS IT HERE
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    val_ds = InSARDataset(
        root_dir=val_dir,
        transform=val_transform,
        exclude_idx=exclude_idx,
        include_only_idx=include_only_idx,
        custom_features=custom_features # <-- PASS IT HERE
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    return train_loader, val_loader

def check_accuracy(loader, model, device="cuda"):
    num_correct = 0
    num_pixels = 0
    dice_score = 0
    
    tp = 0
    fp = 0
    fn = 0
    tn = 0
    
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device).unsqueeze(1)

            preds = torch.sigmoid(model(x))
            preds = (preds > 0.5).float()
            
            num_correct += (preds == y).sum().item()
            num_pixels += torch.numel(preds)
            dice_score += ((2 * (preds * y).sum()) / ((preds + y).sum() + 1e-8)).item()
            
            tp += (preds * y).sum().item()
            fp += (preds * (1 - y)).sum().item()
            fn += ((1 - preds) * y).sum().item()
            tn += ((1 - preds) * (1 - y)).sum().item()

    accuracy = (num_correct / num_pixels) * 100
    avg_dice = dice_score / len(loader)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1_score = 2 * (precision * recall) / (precision + recall + 1e-8)
    
    total_pixels = tp + fp + fn + tn
    p_o = (tp + tn) / (total_pixels + 1e-8)
    p_e = ((tp + fn) * (tp + fp) + (tn + fp) * (tn + fn)) / (total_pixels**2 + 1e-8)
    kappa = (p_o - p_e) / (1 - p_e + 1e-8)

    print(f"Got {num_correct}/{num_pixels} with acc {accuracy:.2f}%")
    print(f"Dice score (batch-averaged): {avg_dice:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")
    print(f"Cohen's Kappa: {kappa:.4f}")
    
    model.train()
    
    return {
        "accuracy": accuracy,
        "dice": avg_dice,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
        "kappa": kappa
    }

def save_predictions_as_imgs(
    loader, 
    model, 
    folder="saved_images/", 
    device="cuda", 
    output_confidence=OUTPUT_CONFIDENCE_SCORES
):
    model.eval()
    if not os.path.exists(folder):
        os.makedirs(folder)
        
    for idx, (x, y) in enumerate(loader):
        x = x.to(device=device)
        with torch.no_grad():
            preds = torch.sigmoid(model(x))
            
            # Only convert to binary 0/1 if output_confidence is False
            if not output_confidence:
                preds = (preds > 0.5).float()
        
        # Save Prediction (Original grid layout)
        torchvision.utils.save_image(preds, f"{folder}/pred_{idx}.png")
        # Save Ground Truth (Original grid layout)
        torchvision.utils.save_image(y.float().unsqueeze(1), f"{folder}/true_{idx}.png")
        
        # Save each predicted result separately
        for b in range(preds.shape[0]):
            dataset_idx = idx * loader.batch_size + b
            
            if hasattr(loader.dataset, 'mask_files') and dataset_idx < len(loader.dataset.mask_files):
                mask_filename = loader.dataset.mask_files[dataset_idx]
                base_tile = mask_filename.rsplit('_mask.png', 1)[0]
                individual_name = f"{base_tile}_pred.png"
            else:
                individual_name = f"pred_{idx}_sample_{b}.png"
                
            torchvision.utils.save_image(preds[b], f"{folder}/{individual_name}")
        
        if x.shape[1] > 6:
            torchvision.utils.save_image(x[:, 6:7, :, :], f"{folder}/input_amp_{idx}.png")

    model.train()

In [32]:
def train_fn(loader, model, optimizer, loss_fn, scaler):
    loop = tqdm(loader)
    total_loss = 0.0
    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(device=DEVICE)
        targets = targets.float().unsqueeze(1).to(device=DEVICE)

        # forward
        with torch.cuda.amp.autocast():
            predictions = model(data)
            loss = loss_fn(predictions, targets)

        # backward
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        # update tqdm loop
        loop.set_postfix(loss=loss.item())
        
    # Return the average loss for this epoch
    return total_loss / len(loader)

def main():
    train_transform = A.Compose([
        A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH),
        ToTensorV2(),
    ])

    val_transform = A.Compose([
        A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH),
        ToTensorV2(),
    ])

    suffixes = [
        "coh_vh_diff", "coh_vv_diff", 
        "coh_vh_nonflood", "coh_vh_flood", "coh_vv_nonflood", "coh_vv_flood",
        "amp_vh_reference", "amp_vh_secondary", "amp_vv_reference", "amp_vv_secondary",
        "dem", "land_use", "buildings",
        "inlets", "outlets", "open_drains", "gravity_mains", "roads", "slope", "streamflow_direction"
    ]

    # --- DYNAMIC EXPERIMENT SETUP ---
    if RUN_ABLATION_STUDY:
        print("--> Running in ABLATION MODE (20 separate feature profiles) <--")
        experiments = list(enumerate(suffixes))
        in_channels = 19
    elif RUN_SINGLE_FEATURE_STUDY:
        print("--> Running in SINGLE-FEATURE MODE (20 separate runs) <--")
        experiments = list(enumerate(suffixes))  
        in_channels = 1
    elif RUN_CUSTOM_FEATURE_STUDY:
        print(f"--> Running in CUSTOM FEATURE STUDY MODE (Subset: {CUSTOM_FEATURES}) <--")
        experiments = [(None, "custom_feature_subset")]
        in_channels = len(CUSTOM_FEATURES)
    else:
        print("--> Running in STANDARD MODE (1 feature profile, all 20 layers included) <--")
        experiments = [(None, "all_features_included")]
        in_channels = 20

    ## !!!!!!!!!!!!!!!!!!!!! ##
    NUM_ROUNDS = 1
    
    # Create the dedicated directory for storing .tar model checkpoints cleanly
    checkpoint_folder = "saved_models"
    os.makedirs(checkpoint_folder, exist_ok=True)
    
    # Trackers for performance metrics
    final_summary = {}
    all_individual_runs_list = []
    
    # NEW: Master list to store metrics for EVERY epoch across all experiments/rounds
    all_epochs_records = []

    # --- MAIN EXECUTION LOOP ---
    for run_idx, (idx, label_name) in enumerate(experiments):
        print(f"\n{'='*60}")
        if RUN_ABLATION_STUDY:
            print(f"RUN {run_idx + 1}/20 | EXCLUDING FEATURE LAYER: {label_name}")
            exclude_idx = idx
            include_only_idx = None
        elif RUN_SINGLE_FEATURE_STUDY:
            print(f"RUN {run_idx + 1}/20 | TRAINING WITH ONLY FEATURE LAYER: {label_name}")
            exclude_idx = None
            include_only_idx = idx
        elif RUN_CUSTOM_FEATURE_STUDY:
            print(f"CUSTOM RUN | USING {in_channels} FEATURES: {CUSTOM_FEATURES}")
            exclude_idx = None
            include_only_idx = None
        else:
            print(f"STANDARD RUN | USING ALL {in_channels} FEATURES")
            exclude_idx = None
            include_only_idx = None
        print(f"{'='*60}")

        # --- DYNAMIC EMBEDDING INDEX TRACKING ---
        if RUN_SINGLE_FEATURE_STUDY:
            current_land_use_idx = 0 if label_name == "land_use" else None
        elif RUN_ABLATION_STUDY:
            if label_name == "land_use":
                current_land_use_idx = None
            else:
                current_suffixes = [s for i, s in enumerate(suffixes) if i != exclude_idx]
                current_land_use_idx = current_suffixes.index("land_use")
        elif RUN_CUSTOM_FEATURE_STUDY:
            current_land_use_idx = CUSTOM_FEATURES.index("land_use") if "land_use" in CUSTOM_FEATURES else None
        else:
            current_land_use_idx = suffixes.index("land_use")

        # Arrays to collect peak historic values for each of the runs
        round_accuracies = []
        round_dices = []
        round_precisions = []
        round_recalls = []
        round_f1s = []
        round_kappas = []

        # --- MULTI-ROUND EVALUATION LOOP ---
        for round_idx in range(NUM_ROUNDS):
            print(f"\n>>> Starting Round {round_idx + 1}/{NUM_ROUNDS} for feature profile: {label_name} <<<")
            
            current_seed = 42 + round_idx
            seed_everything(seed=current_seed)

            model = UNET(
                in_channels=in_channels, 
                out_channels=1,
                land_use_idx=current_land_use_idx,
                num_land_use_classes=256,  
                embedding_dim=8           
            ).to(DEVICE)
            
            loss_fn = nn.BCEWithLogitsLoss()
            optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
            scaler = torch.cuda.amp.GradScaler()

            train_loader, val_loader = get_loaders(
                TRAIN_DIR, TRAIN_DIR,
                VAL_DIR, VAL_DIR,
                BATCH_SIZE,
                train_transform,
                val_transform,
                NUM_WORKERS,
                PIN_MEMORY,
                exclude_idx=exclude_idx,
                include_only_idx=include_only_idx,
                custom_features=CUSTOM_FEATURES if RUN_CUSTOM_FEATURE_STUDY else None
            )

            best_f1 = -1.0
            best_metrics_this_round = None
            
            run_folder = f"saved_images/run_{label_name}_round_{round_idx + 1}"
            os.makedirs(run_folder, exist_ok=True)

            # --- INNER LOOP: FULL EPOCH SESSIONS ---
            for epoch in range(NUM_EPOCHS):
                print(f"Target: {label_name} | Round {round_idx + 1}/{NUM_ROUNDS} | Epoch [{epoch+1}/{NUM_EPOCHS}]")
                
                # Capture average loss returned from train_fn
                epoch_loss = train_fn(train_loader, model, optimizer, loss_fn, scaler)

                # Validate metrics
                metrics = check_accuracy(val_loader, model, device=DEVICE)
                current_f1 = metrics["f1_score"]

                # NEW: Record stats for EVERY epoch directly into the tracking list
                all_epochs_records.append({
                    "Feature Profile": label_name,
                    "Round Index": round_idx + 1,
                    "Epoch": epoch + 1,
                    "Loss": round(epoch_loss, 6),
                    "Accuracy (%)": round(metrics["accuracy"], 2),
                    "F1 Score": round(current_f1, 4)
                })

                # Monitor peak threshold checkpoints within this round
                if current_f1 > best_f1:
                    best_f1 = current_f1
                    best_metrics_this_round = metrics
                    
                    print(f"*** New peak F1 Score for Round {round_idx + 1}: {best_f1:.4f} at Epoch {epoch + 1}! ***")

                    checkpoint = {
                        "state_dict": model.state_dict(),
                        "optimizer": optimizer.state_dict(),
                    }
                    
                    checkpoint_filename = os.path.join(checkpoint_folder, f"best_model_{label_name}_round_{round_idx + 1}.pth.tar")
                    save_checkpoint(checkpoint, filename=checkpoint_filename) #full save
                    #save_checkpoint({"state_dict": model.state_dict()}, filename="best_model.pth.tar") #only model weights

                    save_predictions_as_imgs(
                        val_loader, model, folder=run_folder, device=DEVICE
                    )

            # Save the optimal findings recorded in this independent seed run
            round_accuracies.append(best_metrics_this_round["accuracy"])
            round_dices.append(best_metrics_this_round["dice"])
            round_precisions.append(best_metrics_this_round["precision"])
            round_recalls.append(best_metrics_this_round["recall"])
            round_f1s.append(best_metrics_this_round["f1_score"])
            round_kappas.append(best_metrics_this_round["kappa"])

            all_individual_runs_list.append({
                "Feature Profile": label_name,
                "Round Index": round_idx + 1,
                "Random Seed": current_seed,
                "Accuracy (%)": round(best_metrics_this_round["accuracy"], 2),
                "Dice Score": round(best_metrics_this_round["dice"], 4),
                "Precision": round(best_metrics_this_round["precision"], 4),
                "Recall": round(best_metrics_this_round["recall"], 4),
                "F1 Score": round(best_metrics_this_round["f1_score"], 4),
                "Cohen's Kappa": round(best_metrics_this_round["kappa"], 4)
            })

        # --- AGGREGATION PHASE ---
        final_summary[label_name] = {
            "Mean Acc (%)": round(np.mean(round_accuracies), 2),
            "Std Acc (%)": round(np.std(round_accuracies), 2),
            "Mean Dice": round(np.mean(round_dices), 4),
            "Std Dice": round(np.std(round_dices), 4),
            "Mean Precision": round(np.mean(round_precisions), 4),
            "Mean Recall": round(np.mean(round_recalls), 4),
            "Mean F1 Score": round(np.mean(round_f1s), 4),
            "Std F1 Score": round(np.std(round_f1s), 4),
            "Mean Kappa": round(np.mean(round_kappas), 4)
        }

    # --- EXPORT & PRINT AGGREGATED SUMMARY TABLE ---
    print(f"\n\n{'='*35} FINAL PERFORMANCE SUMMARY (AVERAGED) {'='*35}")
    df_summary = pd.DataFrame.from_dict(final_summary, orient='index')
    print(df_summary.to_string())
    
    if RUN_ABLATION_STUDY:
        summary_filename = "ablation_study_averaged_results.csv"
        individual_filename = "ablation_study_all_individual_runs.csv"
        epoch_csv_filename = "ablation_study_epoch_metrics.csv"
    elif RUN_SINGLE_FEATURE_STUDY:
        summary_filename = "single_feature_study_averaged_results.csv"
        individual_filename = "single_feature_study_all_individual_runs.csv"
        epoch_csv_filename = "single_feature_study_epoch_metrics.csv"
    elif RUN_CUSTOM_FEATURE_STUDY:
        summary_filename = "custom_feature_study_averaged_results.csv"
        individual_filename = "custom_feature_study_all_individual_runs.csv"
        epoch_csv_filename = "custom_feature_study_epoch_metrics.csv"
    else:
        summary_filename = "standard_run_averaged_results.csv"
        individual_filename = "standard_run_all_individual_runs.csv"
        epoch_csv_filename = "standard_run_epoch_metrics.csv"
        
    df_summary.to_csv(summary_filename)
    print(f"\nAveraged summary results exported to '{summary_filename}'")

    # --- EXPORT GRANULAR RUN-BY-RUN SHEET ---
    df_individual = pd.DataFrame(all_individual_runs_list)
    df_individual.to_csv(individual_filename, index=False)
    print(f"Granular individual run data successfully exported to '{individual_filename}'")
    
    # NEW: Export the logged epoch track sheets into a clean CSV file
    if all_epochs_records:
        df_epochs = pd.DataFrame(all_epochs_records)
        df_epochs.to_csv(epoch_csv_filename, index=False)
        print(f"All epoch metrics successfully saved to CSV file: '{epoch_csv_filename}'")
    
if __name__ == "__main__":
    main()

--> Running in CUSTOM FEATURE STUDY MODE (Subset: ['coh_vh_diff', 'coh_vv_diff', 'dem', 'coh_vh_nonflood', 'coh_vh_flood', 'coh_vv_nonflood', 'coh_vv_flood', 'amp_vh_reference', 'amp_vh_secondary', 'amp_vv_reference', 'amp_vv_secondary', 'slope', 'streamflow_direction']) <--

CUSTOM RUN | USING 13 FEATURES: ['coh_vh_diff', 'coh_vv_diff', 'dem', 'coh_vh_nonflood', 'coh_vh_flood', 'coh_vv_nonflood', 'coh_vv_flood', 'amp_vh_reference', 'amp_vh_secondary', 'amp_vv_reference', 'amp_vv_secondary', 'slope', 'streamflow_direction']

>>> Starting Round 1/1 for feature profile: custom_feature_subset <<<


/tmp/ipykernel_2236391/3389273574.py:137: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Target: custom_feature_subset | Round 1/1 | Epoch [1/150]


  0%|          | 0/1 [00:00<?, ?it/s]/tmp/ipykernel_2236391/3389273574.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 1/1 [00:00<00:00,  1.07it/s, loss=0.786]


Got 138562/303372 with acc 45.67%
Dice score (batch-averaged): 0.6271
Precision: 0.4567
Recall: 1.0000
F1 Score: 0.6271
Cohen's Kappa: 0.0001
*** New peak F1 Score for Round 1: 0.6271 at Epoch 1! ***
=> Saving checkpoint
Target: custom_feature_subset | Round 1/1 | Epoch [2/150]


...

Got 264214/303372 with acc 87.09%
Dice score (batch-averaged): 0.8440
Precision: 0.9419
Recall: 0.7645
F1 Score: 0.8440
Cohen's Kappa: 0.7359
Target: custom_feature_subset | Round 1/1 | Epoch [149/150]


100%|██████████| 1/1 [00:00<00:00,  1.11it/s, loss=0.0376]


Got 265309/303372 with acc 87.45%
Dice score (batch-averaged): 0.8500
Precision: 0.9363
Recall: 0.7782
F1 Score: 0.8500
Cohen's Kappa: 0.7437
Target: custom_feature_subset | Round 1/1 | Epoch [150/150]


100%|██████████| 1/1 [00:00<00:00,  1.11it/s, loss=0.0371]


Got 263740/303372 with acc 86.94%
Dice score (batch-averaged): 0.8428
Precision: 0.9356
Recall: 0.7667
F1 Score: 0.8428
Cohen's Kappa: 0.7329


=================================== FINAL PERFORMANCE SUMMARY (AVERAGED) ===================================
                       Mean Acc (%)  Std Acc (%)  Mean Dice  Std Dice  Mean Precision  Mean Recall  Mean F1 Score  Std F1 Score  Mean Kappa
custom_feature_subset         88.95          0.0     0.8811       0.0          0.8663       0.8964         0.8811           0.0      0.7779

Averaged summary results exported to 'custom_feature_study_averaged_results.csv'
Granular individual run data successfully exported to 'custom_feature_study_all_individual_runs.csv'
All epoch metrics successfully saved to CSV file: 'custom_feature_study_epoch_metrics.csv'
